In [ ]:
from time import sleep

from distributed_processing.redis_connector import RedisConnector
from distributed_processing.utils import redisnode
from distributed_processing.worker import Worker

In [ ]:
REDIS_HOST = "localhost"
REDIS_PORT = 6379
REDIS_DB = 0
NAMESPACE = "tasks"
NODE_ID = "node_1"  # the node's queue name; clients target it with queue=NODE_ID

In [ ]:
# Worker constructor. It runs inside a subprocess started with `spawn`, so it
# builds its own connector. Defined in this notebook, it is dill-serialized by
# value into the subprocess, together with the globals it uses (REDIS_HOST...).
def worker1(worker_id=None):
    server = Worker(
        RedisConnector(REDIS_HOST, REDIS_PORT, REDIS_DB, NAMESPACE), worker_id=worker_id
    )

    def info():
        rq = {}
        for k, v in server.requests_queues.items():
            rq[k] = sorted(v[0].keys())
        return server.worker_id, rq

    server.add_requests_queue(server.worker_id, {"info": info})

    def add(x, y):
        return x + y

    def mul(x, y):
        return x * y

    def div(x, y):
        return x / y

    def lista(x, y):
        return [x, y]

    def tupla(x, y):
        return (x, y)

    def dic(x, y):
        return {"a": x, "b": [x, y]}

    func_dict1 = {
        "add": add,
        "mul": mul,
        "div": div,
        "lista": lista,
        "tupla": tupla,
        "dic": dic,
        "sleep": sleep,
    }

    server.add_requests_queue("cola_1", func_dict1)

    def hola(nombre, calificativo="listo"):
        return f"Hola {nombre}, eres muy {calificativo}"

    server.add_requests_queue("cola_2", {"hola": hola})
    # SECURITY: eval_py_function runs any code sent by clients.
    server.add_python_eval()
    server.update_methods_registry()
    return server

In [ ]:
# The node: a master Worker that starts, lists and kills worker subprocesses
# on request (create_worker, list_processes, kill_process, kill_processes,
# kill_all_processes, cleanup). clean=True wipes the whole namespace first.
master = redisnode(
    REDIS_HOST,
    REDIS_PORT,
    REDIS_DB,
    NAMESPACE,
    clean=False,
    worker_id=NODE_ID,
    workers_constructors={"worker1": worker1},
)

In [ ]:
# Optional: start two workers before serving. The same methods are available
# to clients by RPC (see redis_client_node.ipynb). Each call waits until the
# subprocess reports its worker id and returns (pid, worker_type, worker_id).
for _ in range(2):
    print(master.exec_method("create_worker", ["worker1"], queue=NODE_ID))

In [ ]:
master.exec_method("list_processes", queue=NODE_ID)

In [ ]:
# Serve the node methods. run() stops the cell on the first connector error;
# interrupt the kernel to stop it. The workers are killed when this process
# exits (atexit), or by themselves if it is killed hard.
master.run()
# master.run_forever(backoff=(1, 60))

In [ ]:
# After interrupting run(): kill every worker and close the master.
master.exec_method("cleanup", queue=NODE_ID)